# Crisis Period Analysis – Portfolio Replication
### Machine Learning for Fintech – Politecnico di Milano

---

This notebook evaluates the robustness of all fitted replication models (**OLS, Ridge, Lasso, ElasticNet, Kalman Filter**) during three major macro-financial stress episodes contained in the dataset sample.

> **Prerequisites:** this notebook must be run after the modeling notebook, which produces the result dictionaries `ols_result`, `best_ridge_full`, `best_lasso_full`, `best_en_full`, and `kf_best`, as well as the helper functions `calcola_metriche` and `calcola_replication_score`.

| Crisis | Period | Main driver |
|--------|--------|-------------|
| **GFC 2008** | Aug 2008 – Mar 2009 | Lehman collapse, global credit crunch |
| **Eurozone 2011** | Jul 2011 – Dec 2011 | BTP-Bund spread, sovereign debt stress |
| **COVID-19 2020** | Feb 2020 – Apr 2020 | Global lockdowns, liquidity crash |

> The 2022 inflation episode is excluded because the dataset ends in April 2021 and would produce no observations for any model.


## 1. Crisis Period Definitions and Helper Function

The helper `metriche_sottoperiodo` slices the OOS backtest results to the requested date range and recomputes all metrics on that sub-period. A minimum of 4 weekly observations is required to return meaningful statistics; otherwise the function returns `None` and the period is skipped in all outputs.

In [ ]:
crisis_periods = {
    'GFC 2008'      : ('2008-08-01', '2009-03-31'),
    'Eurozone 2011' : ('2011-07-01', '2011-12-31'),
    'COVID-19 2020' : ('2020-02-01', '2020-04-30'),
}

CRISIS_COLORS = {
    'GFC 2008'      : '#D62728',
    'Eurozone 2011' : '#FF7F0E',
    'COVID-19 2020' : '#9467BD',
}

MODEL_COLORS = ['steelblue', 'darkorange', 'seagreen', 'crimson', 'purple']

tutti_i_modelli = {
    'OLS'          : ols_result,
    'Ridge'        : best_ridge_full,
    'Lasso'        : best_lasso_full,
    'ElasticNet'   : best_en_full,
    'Kalman Filter': kf_best,
}

def metriche_sottoperiodo(result_dict, start, end):
    """
    Recomputes performance metrics for a specific sub-period.
    Returns None if fewer than 4 observations are available.
    """
    rep_sub = result_dict['replica_returns'].loc[start:end]
    tgt_sub = result_dict['target_returns'].loc[start:end]
    if len(rep_sub) < 4:
        return None
    return calcola_metriche(rep_sub, tgt_sub)

print('Definitions loaded.')
print(f'Models : {list(tutti_i_modelli.keys())}')
print(f'Crises : {list(crisis_periods.keys())}')


## 2. Correlation Matrix by Crisis and Model

The first diagnostic is a quick correlation overview: for each crisis window and each model, we print the OOS correlation between the replica and the Monster Index. This immediately shows which models degrade the most under stress.

In [ ]:
# ── Quick correlation overview (printed table) ──
print('OOS Correlation during crisis periods\n')
header = f"{'Crisis':<22}"
for nome in tutti_i_modelli:
    header += f'{nome:>15}'
print(header)
print('-' * (22 + 15 * len(tutti_i_modelli)))

for crisi, (start, end) in crisis_periods.items():
    row = f'{crisi:<22}'
    for nome, res in tutti_i_modelli.items():
        m = metriche_sottoperiodo(res, start, end)
        row += f"{m['correlation']:>15.4f}" if m else f"{'N/A':>15}"
    print(row)


## 3. Full Metrics Table

For each (crisis, model) pair we compute four metrics:

- **Corr.** — Pearson correlation between replica and target returns
- **TE (%)** — annualized tracking error (lower = closer replication)
- **IR** — information ratio (active return per unit of active risk)
- **MDD (%)** — maximum drawdown of the replica during the crisis window


In [ ]:
rows = []
for crisi, (start, end) in crisis_periods.items():
    for nome, res in tutti_i_modelli.items():
        m = metriche_sottoperiodo(res, start, end)
        if m is None:
            continue
        rows.append({
            'Crisis' : crisi,
            'Model'  : nome,
            'Corr.'  : round(m['correlation'], 4),
            'TE (%)' : round(m['tracking_error'] * 100, 2),
            'IR'     : round(m['info_ratio'], 3),
            'MDD (%)': round(m['max_drawdown'] * 100, 2),
        })

crisis_tab = pd.DataFrame(rows).set_index(['Crisis', 'Model'])

display(
    crisis_tab.style
    .format({'Corr.': '{:.4f}', 'TE (%)': '{:.2f}', 'IR': '{:.3f}', 'MDD (%)': '{:.2f}'})
    .background_gradient(subset=['Corr.'], cmap='RdYlGn', vmin=-0.2, vmax=1.0)
    .background_gradient(subset=['TE (%)'], cmap='RdYlGn_r', vmin=0)
    .set_caption('Full Performance Metrics by Crisis Period and Model')
)


## 4. OOS Correlation by Crisis — Visual Comparison

Each panel shows the correlation of each model with the Monster Index during one crisis. The tracking error (TE) is annotated above each bar. A model that maintains high correlation and low TE under stress is preferable for real-world portfolio replication.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for ax, (crisi, (start, end)) in zip(axes, crisis_periods.items()):
    nomi, corrs, tes = [], [], []
    for nome, res in tutti_i_modelli.items():
        m = metriche_sottoperiodo(res, start, end)
        if m is not None:
            nomi.append(nome)
            corrs.append(m['correlation'])
            tes.append(m['tracking_error'] * 100)

    x    = np.arange(len(nomi))
    bars = ax.bar(x, corrs, color=MODEL_COLORS[:len(nomi)],
                 edgecolor='black', linewidth=0.5, alpha=0.85)

    for bar, te in zip(bars, tes):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f'TE={te:.1f}%',
            ha='center', va='bottom', fontsize=8
        )

    ax.set_title(f'{crisi}\n{start[:7]} → {end[:7]}',
                 fontsize=12, fontweight='bold', color=CRISIS_COLORS[crisi])
    ax.set_ylabel('OOS Correlation')
    ax.set_xticks(x)
    ax.set_xticklabels(nomi, rotation=30, ha='right', fontsize=9)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylim(-0.25, 1.15)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('OOS Correlation During Crisis Periods – All Models',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 5. Cumulative Return During Each Crisis

Each panel shows the normalised cumulative return (base = 1 at crisis start) of the Monster Index (solid black) and each replica (dashed). A good replica tracks the drawdown and recovery path of the target closely.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (crisi, (start, end)) in zip(axes, crisis_periods.items()):
    tgt_sub = target_aligned.loc[start:end]
    if len(tgt_sub) < 2:
        ax.set_title(f'{crisi}\n(no data available)')
        continue

    cum_tgt = (1 + tgt_sub).cumprod()
    cum_tgt = cum_tgt / cum_tgt.iloc[0]
    ax.plot(cum_tgt, color='black', linewidth=2.5, label='Monster Index', zorder=5)

    for (nome, res), col in zip(tutti_i_modelli.items(), MODEL_COLORS):
        rep_sub = res['replica_returns'].loc[start:end]
        if len(rep_sub) < 2:
            continue
        cum_rep = (1 + rep_sub).cumprod()
        cum_rep = cum_rep / cum_rep.iloc[0]
        ax.plot(cum_rep, linewidth=1.6, linestyle='--',
                color=col, label=nome, alpha=0.85)

    ax.axhline(1, color='gray', linewidth=0.7, linestyle=':')
    ax.set_title(f'{crisi}\n{start[:7]} → {end[:7]}',
                 fontsize=12, fontweight='bold', color=CRISIS_COLORS[crisi])
    ax.set_ylabel('Cumulative return (base = 1)')
    ax.legend(fontsize=8, loc='lower left', framealpha=0.85)
    ax.grid(True, alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b'%y"))
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('Cumulative Return During Crisis Periods – All Models vs Monster Index',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 6. Replication Score During Crisis Periods

The composite Replication Score combines correlation, tracking error, information ratio, turnover, and relative drawdown into a single number (0–1, higher = better). The heatmap below shows whether the model ranking observed on the full OOS period holds up under stress — a model that drops sharply in the crisis score is not robust enough for production use.

In [ ]:
crisis_score_rows = []

for crisi, (start, end) in crisis_periods.items():
    for nome, res in tutti_i_modelli.items():
        m = metriche_sottoperiodo(res, start, end)
        if m is None:
            continue

        # Turnover in the sub-period — use .get() as a safety fallback
        turn_hist = res.get('turnover_history', pd.Series(dtype=float))
        turn_sub  = turn_hist.loc[start:end] if len(turn_hist) > 0 else pd.Series(dtype=float)
        m['avg_turnover'] = turn_sub.mean() if len(turn_sub) > 0 else res.get('avg_turnover', 0.0)

        score, _ = calcola_replication_score(m)
        crisis_score_rows.append({
            'Crisis' : crisi,
            'Model'  : nome,
            'Score'  : round(score, 4),
            'Corr.'  : round(m['correlation'], 4),
            'TE (%)'  : round(m['tracking_error'] * 100, 2),
        })

crisis_score_df = pd.DataFrame(crisis_score_rows)
pivot = (
    crisis_score_df
    .pivot(index='Crisis', columns='Model', values='Score')
    .reindex(columns=list(tutti_i_modelli.keys()))
    .reindex(list(crisis_periods.keys()))
)

print('Replication Score — Crisis Period x Model:\n')
display(pivot.round(4))

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Heatmap
sns.heatmap(
    pivot, annot=True, fmt='.3f', cmap='RdYlGn',
    vmin=0.3, vmax=0.8, linewidths=0.5,
    ax=axes[0], cbar_kws={'label': 'Replication Score'}
)
axes[0].set_title('Replication Score Heatmap\n(Crisis × Model)', fontsize=12)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Crisis')
axes[0].tick_params(axis='x', rotation=30)

# Score drop: full-OOS score vs crisis score for each model
full_scores = {}
for nome, res in tutti_i_modelli.items():
    sc, _ = calcola_replication_score(res)
    full_scores[nome] = sc

crisis_mean = pivot.mean(axis=0)
drop_df = pd.DataFrame({
    'Full OOS'    : pd.Series(full_scores),
    'Crisis Mean' : crisis_mean,
}).reindex(list(tutti_i_modelli.keys()))
drop_df['Drop'] = drop_df['Full OOS'] - drop_df['Crisis Mean']

x = np.arange(len(drop_df))
w = 0.35
axes[1].bar(x - w/2, drop_df['Full OOS'],    width=w, label='Full OOS',    alpha=0.85, color='steelblue')
axes[1].bar(x + w/2, drop_df['Crisis Mean'], width=w, label='Crisis Mean', alpha=0.85, color='crimson')
axes[1].set_xticks(x)
axes[1].set_xticklabels(drop_df.index, rotation=30, ha='right')
axes[1].set_title('Score Drop: Full OOS vs Average Crisis Score', fontsize=12)
axes[1].set_ylabel('Replication Score')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.25, axis='y')
axes[1].set_ylim(0, 0.9)

# Annotate drop
for i, (nome, row) in enumerate(drop_df.iterrows()):
    axes[1].text(i, max(row['Full OOS'], row['Crisis Mean']) + 0.01,
                 f"Δ={row['Drop']:.3f}", ha='center', fontsize=8, color='black')

plt.tight_layout()
plt.show()

# Key findings
most_robust  = drop_df['Drop'].idxmin()
least_robust = drop_df['Drop'].idxmax()
print(f'Most robust model under stress  : {most_robust}  (score drop = {drop_df.loc[most_robust, "Drop"]:.3f})')
print(f'Least robust model under stress : {least_robust}  (score drop = {drop_df.loc[least_robust, "Drop"]:.3f})')


## 7. Crisis Analysis — Key Takeaways

The analysis above leads to four practical conclusions:

1. **Correlation is not stable across regimes.** A model that scores well on the full OOS sample may correlate much less with the target during acute stress (GFC, COVID). This confirms that static fixed-weight models are fragile.

2. **Tracking error spikes during crises.** Even for the best models, TE increases substantially during volatile periods — meaning replication becomes less precise exactly when precision matters most.

3. **The Replication Score drop reveals real robustness.** A small drop (Δ) between full-OOS score and average crisis score is the key indicator of a model suitable for stress-resilient deployment.

4. **Dynamic weights (Kalman Filter) tend to be more robust.** Because the Kalman Filter updates exposures every week, it can partly adapt to correlation shifts during crises, whereas rolling-window linear models use stale coefficients estimated in calmer periods.
